In [1]:
!pip install discord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.4 MB/s eta 0:00:00


In [4]:
!unzip ./converted_keras.zip

Archive:  ./converted_keras.zip
 extracting: keras_model.h5          
 extracting: labels.txt              


In [7]:
# importamos la libreria PIL: Python Imaging Library
# es necesario para poder abrir la imagen a analizar
import PIL
from PIL import Image, ImageOps
import numpy as np
# Instalamos keras, libreria necesaria para que funcionen
# las redes neuronales
!pip install -q tf-keras==2.19.0 h5py==3.11.0
# Importando tf-keras – una versión de Keras compatible con modelos .h5
import tf_keras as keras
# Importando la función load_model de tf_keras, que nos
# permitira cargar el modelo de google teachable machine
from tf_keras.models import load_model

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 7.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement tensorflow<2.20,>=2.19 (from tf-keras) (from versions: 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow<2.20,>=2.19


In [10]:
model_path = "/content/keras_model.h5"
labels_path = "/content/labels.txt"

def get_class(image_path):
  np.set_printoptions(suppress=True)
  model = load_model(model_path, compile=False)
  class_names = open(labels_path, "r").readlines()
  data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
  image = Image.open(image_path).convert("RGB")
  size = (224, 224)
  image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)
  image_array = np.asarray(image)
  normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1
  data[0] = normalized_image_array
  prediction = model.predict(data)
  print(prediction)
  index = np.argmax(prediction)
  class_name = class_names[index]
  confidence_score = prediction[0][index]
  text1 = f"Class: {class_name[2:]} Confidence Score: {confidence_score}"
  return text1


In [2]:
import discord
from discord.ext import commands
import nest_asyncio
nest_asyncio.apply()

In [11]:
#Definimos los permisos del bot
intents = discord.Intents.default()
intents.message_content = True
#creamos el bot
bot = commands.Bot(command_prefix='$', intents=intents)

@bot.event
async def on_ready():
    print(f'We have logged in as {bot.user}')
#este comando saluda al usuario
#para activarlo se escribe $hello

@bot.command()
async def hello(ctx):
    await ctx.send(f'Hi! I am a bot {bot.user}!')

#escribe tu codigo aqui
@bot.command()
async def check(ctx):
  if ctx.message.attachments:
    for attachment in ctx.message.attachments:
      file_name = attachment.filename
      file_url = attachment.url
      await attachment.save(f"./{file_name}")
      await ctx.send("Imagen guardada con exito")
      await ctx.send(get_class(f"./{file_name}"))
  else:
    await ctx.send("Olvidaste enviar una imagen")


bot.run("")

2026-09-04 01:40:45 INFO     discord.client logging in using static token
2026-09-04 01:40:45 INFO     discord.client logging in using static token
INFO:discord.client:logging in using static token
2026-09-04 01:40:45 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: c0679eda777005f437a5a30faddcffd4).
2026-09-04 01:40:45 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: c0679eda777005f437a5a30faddcffd4).
INFO:discord.gateway:Shard ID None has connected to Gateway (Session ID: c0679eda777005f437a5a30faddcffd4).


We have logged in as bot ia miercoles 7 pm#6861
1/1 [==============================] - 1s 1s/step
[[0.96171755 0.03828253]]
